In [36]:
#SETUP 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [37]:
# Importing Data 


df = pd.read_csv("hf://datasets/processvenue/SARCASM_VS_NON_SARCASM/PROJ_SARCASM_VS_NON_SARCASM_6000_0004(HF).csv")

# Manual data exploration

## Columns types and Initial exploration

Here we are doing initial, data exploration, size of the dataset, types of columns, and duplicated

In [45]:
#----------------------------------------------------------
#Exploring Data 
print("--Data Loaded COLUMNS---")
print(df.columns)

print("--Data Loaded HEAD---")
print(df.head())

print("--Data Loaded INFO---")
print(df.info())

#----------------------------------------------------------
#Example of a sarcastic sentence and a non sarcastic sentence when they agree
print("--Data Loaded Example of a sarcastic sentence---")
print(df[(df["Annotator 1"] == "Sarcastic") & (df["Annotator 2"] == "Sarcastic")].Sentence.iloc[0])
print("--Data Loaded Example of a non sarcastic sentence---")
print(df[(df["Annotator 1"] == "Non-Sarcastic") & (df["Annotator 2"] == "Non-Sarcastic")].Sentence.iloc[0])

print("-- Example of a sentence where annotators disagree---")
print(df[(df["Annotator 1"] == "Sarcastic") & (df["Annotator 2"] == "Non-Sarcastic")].Sentence.iloc[0])

#----------------------------------------------------------
#Exploring duplicated and null values
print("--Data Loaded checking duplicated---")
print(df.duplicated().sum())
print("--Data Loaded checking unique values---")
print(df.Sentence.duplicated().sum())

#we do have duplicated setences, let's check them closely
print("--Data Loaded checking duplicated sentences---")
print(df[df.Sentence.duplicated(keep=False)].sort_values(by="Sentence"))

#in any of the have different labels?
print("--Data Loaded checking labels for duplicated sentences---")
print(df[df.Sentence.duplicated(keep=False)].groupby("Sentence").agg(lambda x: x.unique().tolist()))

# Null count per column
print(df.isnull().sum())

# Total rows containing at least one null value
print("\nRows with any null:")
print(df.isnull().any(axis=1).sum())


--Data Loaded COLUMNS---
Index(['S.No', 'Sentence', 'Annotator 1', 'Annotator 2', 'IAA'], dtype='str')
--Data Loaded HEAD---
   S.No                                           Sentence    Annotator 1  \
0     1  with such eloquent and compelling arguments as...      Sarcastic   
1     2  fantasy,fairy tale, wild imagination, adsurd ,...  Non-Sarcastic   
2     3  'if carrying guns became legal, then police co...      Sarcastic   
3     4  It was a troll. Now that we've activated email...      Sarcastic   
4     5  Poor little poopie. \r\nIf your gay sex agenda...      Sarcastic   

     Annotator 2       IAA  
0      Sarcastic     Agree  
1  Non-Sarcastic     Agree  
2      Sarcastic     Agree  
3  Non-Sarcastic  Disagree  
4      Sarcastic     Agree  
--Data Loaded INFO---
<class 'pandas.DataFrame'>
RangeIndex: 5940 entries, 0 to 5939
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   S.No         5940 non-null   int64
 

#### <span style="color:#6b7280">COMMENTS</span>

On this first data exploration, a few things to consider:
1. We have 5 columns: 
- S.No: reference number of the Sentence
- Sentence 
- Annotator 1: Label the first person gave
- Annotator 2: Label the second person gave
- IAA: if the labels given agree
2. Sentences are really human written-like, and there are spacing and graphical errors that we might need to consider, especially in preprocessing (e.g., 'fantasy,fairy tale, wild imagination, absurd...').
3. We don't have any nulls (in any of the columns).
4. We have *duplicated sentences*, below we deep-dive on it.
5. In total, in the raw dataset, we have 5940 sentences

Few examples to clarify the lexical complexity:
- Sentence when they agreed on sarcasm: "with such eloquent and compelling arguments as this i'm surprised it hasn't happened already."
- Sentence when they agreed on non-sarcasm: "fantasy,fairy tale, wild imagination, adsurd , silly , ridiculous,fabricated .unsupported by one single observation or experiment , unsubstantiated. i do agree we don't speak the same language. no one will explain to me how one steals public information, i am confused. how is work shady? how would you define shady as opposed to rather shady when it refers to work? i thought bookmaking etc. was shady but i am old fashioned i guess. please educate me. is this a meaningfull reprimand coming from someone who believes one can observe a mind? you still have refused to explain that and i was so anxious to hear it. please go on."
- Sentence when they disagreed on sarcasm: "It was a troll. Now that we've activated email response, it could take another look or expand some mail list."



### Handling *duplicated sentences*

In [39]:
# Getting one manual example of a sentence to check if the labels are correct
print(df[df.Sentence.str.contains("Life threatening condition that is always phy")])

# Dropping duplicated sentences 
df_n_duplicated = df[~df.duplicated(subset="Sentence", keep=False)].copy()
print("--Data Loaded checking duplicated sentences after filtering---")
print(df[df.duplicated(subset="Sentence", keep=False)].shape[0])
print(df_n_duplicated.shape[0])

# Checking duplicated sentences after filtering - SANITY CHECK
print("--Data Loaded checking duplicated sentences after filtering---")
print(df_n_duplicated[df_n_duplicated.Sentence.duplicated(keep=False)].sort_values(by="Sentence"))

      S.No                                           Sentence    Annotator 1  \
1423  1424  'Life threatening condition that is always phy...  Non-Sarcastic   
2716  2757  'Life threatening condition that is always phy...  Non-Sarcastic   
3280  3341  'Life threatening condition that is always phy...  Non-Sarcastic   

        Annotator 2       IAA  
1423  Non-Sarcastic     Agree  
2716      Sarcastic  Disagree  
3280  Non-Sarcastic     Agree  
--Data Loaded checking duplicated sentences after filtering---
238
5702
--Data Loaded checking duplicated sentences after filtering---
Empty DataFrame
Columns: [S.No, Sentence, Annotator 1, Annotator 2, IAA]
Index: []


#### <span style="color:#6b7280">COMMENTS</span>

As we have just a few duplicated sentences (238 of 5940), and sometimes they disagree (same annotator, for the same sentence can have two different opinions), we decided to drop those duplicates.

Example of disagreement:

- Setence N 1424, 2757 and 3280 are the same: "'Life threatening condition that is always physically harmful"? What a giant load of steamy BS. Rarely is pregnancy physically harmful and even rarer is it life threatening. There are over 6 Billion people on this earth today. If motherhood was so damn dangerous, don't you think the population would be a lot lower, even if it was just due to mothers keeling over after giving birth? Where do you get this BS from, Planned Parenthood?"

- For sentence 1424, annotator 1 believes there is no sarcasm, while in 2757 the opinion is inverted and they believe there is sarcasm.

## Numerical exploration

Here we explore more numerically, distribution of the label ["Sacarsm","Non-Sacarsm"] and the agreement as we have two opinions for labelling sarcasm.

### First Step: class imbanlancess 

In [16]:
#distribution of agreements
print("--Distribution of agreements---")
print(df_n_duplicated['IAA'].value_counts())


#For those where they disagree, what is the distribution of both annotators (is there any possibility of bias)?
print("--Distribution of annotators for disagreements---")
print(df_n_duplicated[df_n_duplicated['IAA'] == "Disagree"]['Annotator 1'].value_counts())
print(df_n_duplicated[df_n_duplicated['IAA'] == "Disagree"]['Annotator 2'].value_counts())
#annotator 1 Is a bit more likely to label as sarcastic than annotator 2, but the difference is not huge.

print("--Distribution of labels (WHEN THEY AGREE)---")
print(df_n_duplicated[df_n_duplicated['IAA'] == "Agree"]['Annotator 1'].value_counts())
#in percentage terms
print("--Distribution of labels (WHEN THEY AGREE) in percentage---")
print(df_n_duplicated[df_n_duplicated['IAA'] == "Agree"]['Annotator 1'].value_counts(normalize=True))

--Distribution of agreements---
IAA
Agree       3634
Disagree    2068
Name: count, dtype: int64
--Distribution of annotators for disagreements---
Annotator 1
Sarcastic        1091
Non-Sarcastic     977
Name: count, dtype: int64
Annotator 2
Non-Sarcastic    1091
Sarcastic         977
Name: count, dtype: int64
--Distribution of labels (WHEN THEY AGREE)---
Annotator 1
Sarcastic        2283
Non-Sarcastic    1351
Name: count, dtype: int64
--Distribution of labels (WHEN THEY AGREE) in percentage---
Annotator 1
Sarcastic        0.628233
Non-Sarcastic    0.371767
Name: proportion, dtype: float64


#### <span style="color:#6b7280">COMMENTS</span>

We do have more sarcastic (2283) examples than non-sarcastic examples (1351), but the difference is not huge (62% vs 38%). This is good for training a model, as it will not be too biased towards one class, and if needed can be easily handled with common techniques.

<span style="color:darkorange"><b>Obs:</b> Analysis here is made only in cases where both annotators agree, since these are the labels we are most confident about.</span>

### Second Step: agreement and sacarsm

Here, we gonna wanna have a first impression of two hypothesis: 
- Are longer sentencer more difficult to agree on? 
- Are longer setencers less sarcastic? 


For doing so, we are comparing sacarsm label and agreement with sentence size (here given by numer of tokens).

In [17]:
#Features to compare sarcastic vs non sarcastic sentences
df_n_duplicated["char_length"] = df_n_duplicated["Sentence"].str.len()
df_n_duplicated["token_length"] = df_n_duplicated["Sentence"].str.split().str.len()


#Comparing the agreements vs disagreements in terms of token length, 
# to see if there is any difference in the length of the sentences that they agree on vs the ones they disagree on.
iaa_comparison = (
    df_n_duplicated
    .groupby("IAA")["token_length"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        min="min",
        max="max"
    )
    .round(2)
)

print(iaa_comparison)


#only for the ones they agree, 
df_agree = df_n_duplicated[df_n_duplicated["IAA"] == "Agree"]
# let's compare the token length distribution for sarcastic vs non sarcastic sentences, 
# to see if there is any difference in the length of the sentences that are sarcastic vs non sarcastic.

label_comparison = (
    df_agree
    .groupby("Annotator 1")["token_length"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        min="min",
        max="max"
    )
    .round(2)
)

print(label_comparison)


             n   mean  median    std   p25   p75  min  max
IAA                                                       
Agree     3634  34.90    28.5  22.54  18.0  45.0   10  149
Disagree  2068  36.61    30.0  23.51  19.0  47.0   10  147
                  n   mean  median    std   p25   p75  min  max
Annotator 1                                                    
Non-Sarcastic  1351  39.70    34.0  24.76  21.0  52.0   10  149
Sarcastic      2283  32.05    26.0  20.60  17.0  41.0   10  141


In [40]:
# Final DF after EDA Cleaning 
df_agree = df_n_duplicated[df_n_duplicated['IAA'] == 'Agree'].copy()

#### <span style="color:#6b7280">COMMENTS</span>

1. As we can see from the first table, the distribution of agreement doesn't seem to depend too much on the number of tokens, as both distributions look very similar (e.g. median 28.5 vs 30).
2. From the second table we see that usually non-sarcastic sentences are a little longer (median: 34 vs 26), but the difference is not huge, so no different treatment seems to be needed here.*

<span style="color:darkorange"><b>*Obs:</b> Analysis here is made only in cases where both annotators agree, since these are the labels we are most confident about.</span>

#### <span style="color:#6b7280"> FINAL COMMENTS FOR EDA SECTION</span>

The final Dataframe that we will use will the one filtered from duplicated (238 rows dropped) and where both annotator agree (2068 rows dropped), final df has 3634 rows, with 1351 Non-sarcastic and 2283 sarcarstic senteces
